In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
import plotly.graph_objects as go

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from parentHelpers import *
# from loanPipelineHelpers import *
# from regression import *
# from helperProjectDriver import *
# from helperProjectAlamo import *
from databaseHelpers import *

In [2]:
queries_dict = {
    "all": """
            SELECT 
                    *
                ,	CASE WHEN SUM_BOM_bal > 0 THEN default_amt / SUM_BOM_bal ELSE 0 END AS MDR
                ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (default_amt / SUM_BOM_bal), 12)) * 100 ELSE 0 END AS CDR
                ,	CASE WHEN SUM_BOM_bal > 0 THEN prepayment / prepay_den ELSE 0 END AS SMM
                ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (prepayment / prepay_den), 12)) * 100 ELSE 0 END AS CPR
                ,   CASE WHEN SUM_BOM_bal > 0 THEN SUM_DELIN_amt / SUM_BOM_bal * 100 ELSE 0 END AS delinquency_pct
            FROM
            (
                SELECT
                        DEAL
            		,	'ALL' AS Loan_Type
            		,	'ALL' AS FICO_bkt
                    ,	months_since_deal_clost
                    ,	RPT_DATE
                    ,	COUNT(*) AS num_obs
                    ,	SUM(default_amount) AS default_amt
                    ,	SUM(BOM_BAL_clean) AS SUM_BOM_bal
                    ,	SUM(CURBALANCE_clean) AS SUM_EOM_bal
                    ,	SUM(BOM_BAL_clean * FICO) / NULLIF(SUM(BOM_BAL_clean),0) AS WA_FICO
                    ,	SUM(BOM_BAL_clean * AI_FICO_AT_LOAN_ORIGINATION) / NULLIF(SUM(BOM_BAL_clean), 0) AS WA_ORIG_FICO
                    ,	SUM(BOM_BAL_clean - CURBALANCE_clean - default_amount - Expected_Prin) AS prepayment
                    ,	SUM(BOM_BAL_clean - Expected_Prin) AS prepay_den
                    ,   SUM(delin_amount) AS SUM_DELIN_amt
                FROM
                    #cleanTable
                WHERE DEAL IN ('CMHA19M1', 'CMHA22M1', 'CMHA24M1')
                GROUP BY 
                DEAL, 
            --	Loan_Type, 
            --	FICO_bkt,
            --	Age_Bucket, 
                months_since_deal_clost,
                RPT_DATE
            ) t
            WHERE months_since_deal_clost > 0
            AND (RPT_DATE < '2020-03-01' OR RPT_DATE > '2021-12-31')
            ORDER BY 
            DEAL, 
            Loan_Type, 
            FICO_bkt,
            months_since_deal_clost,
            RPT_DATE
            """

,   "loans_types_only": """
                        SELECT 
                                *
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN default_amt / SUM_BOM_bal ELSE 0 END AS MDR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (default_amt / SUM_BOM_bal), 12)) * 100 ELSE 0 END AS CDR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN prepayment / prepay_den ELSE 0 END AS SMM
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (prepayment / prepay_den), 12)) * 100 ELSE 0 END AS CPR
                            ,   CASE WHEN SUM_BOM_bal > 0 THEN SUM_DELIN_amt / SUM_BOM_bal * 100 ELSE 0 END AS delinquency_pct
                        FROM
                        (
                            SELECT
                                    DEAL
                        		,	Loan_Type
                        		,	'ALL' AS FICO_bkt
                                ,	months_since_deal_clost
                                ,	RPT_DATE
                                ,	COUNT(*) AS num_obs
                                ,	SUM(default_amount) AS default_amt
                                ,	SUM(BOM_BAL_clean) AS SUM_BOM_bal
                                ,	SUM(CURBALANCE_clean) AS SUM_EOM_bal
                                ,	SUM(BOM_BAL_clean * FICO) / NULLIF(SUM(BOM_BAL_clean),0) AS WA_FICO
                                ,	SUM(BOM_BAL_clean * AI_FICO_AT_LOAN_ORIGINATION) / NULLIF(SUM(BOM_BAL_clean), 0) AS WA_ORIG_FICO
                                ,	SUM(BOM_BAL_clean - CURBALANCE_clean - default_amount - Expected_Prin) AS prepayment
                                ,	SUM(BOM_BAL_clean - Expected_Prin) AS prepay_den
                                ,   SUM(delin_amount) AS SUM_DELIN_amt
                            FROM
                                #cleanTable
                            WHERE DEAL IN ('CMHA19M1', 'CMHA22M1', 'CMHA24M1')
                            GROUP BY 
                            DEAL, 
                        	Loan_Type, 
                        --	FICO_bkt,
                            months_since_deal_clost,
                            RPT_DATE
                        ) t
                        WHERE months_since_deal_clost > 0
                        AND (RPT_DATE < '2020-03-01' OR RPT_DATE > '2021-12-31')
                        ORDER BY 
                        DEAL, 
                        Loan_Type,  
                        FICO_bkt,
                        months_since_deal_clost,
                        RPT_DATE
                        """

    ,   "fico_bkt_only": """
                        SELECT 
                                *
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN default_amt / SUM_BOM_bal ELSE 0 END AS MDR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (default_amt / SUM_BOM_bal), 12)) * 100 ELSE 0 END AS CDR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN prepayment / prepay_den ELSE 0 END AS SMM
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (prepayment / prepay_den), 12)) * 100 ELSE 0 END AS CPR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN SUM_DELIN_amt / SUM_BOM_bal * 100 ELSE 0 END AS delinquency_pct
                        FROM
                        (
                            SELECT
                                    DEAL
                        		,	'ALL' AS Loan_Type
                        		,	FICO_bkt
                                ,	months_since_deal_clost
                                ,	RPT_DATE
                                ,	COUNT(*) AS num_obs
                                ,	SUM(default_amount) AS default_amt
                                ,	SUM(BOM_BAL_clean) AS SUM_BOM_bal
                                ,	SUM(CURBALANCE_clean) AS SUM_EOM_bal
                                ,	SUM(BOM_BAL_clean * FICO) / NULLIF(SUM(BOM_BAL_clean),0) AS WA_FICO
                                ,	SUM(BOM_BAL_clean * AI_FICO_AT_LOAN_ORIGINATION) / NULLIF(SUM(BOM_BAL_clean), 0) AS WA_ORIG_FICO
                                ,	SUM(BOM_BAL_clean - CURBALANCE_clean - default_amount - Expected_Prin) AS prepayment
                                ,	SUM(BOM_BAL_clean - Expected_Prin) AS prepay_den
                                ,   SUM(delin_amount) AS SUM_DELIN_amt
                            FROM
                                #cleanTable
                            WHERE DEAL IN ('CMHA19M1', 'CMHA22M1', 'CMHA24M1')
                            GROUP BY 
                            DEAL, 
                        --	Loan_Type, 
                        	FICO_bkt,
                            months_since_deal_clost,
                            RPT_DATE
                        ) t
                        WHERE months_since_deal_clost > 0
                        AND (RPT_DATE < '2020-03-01' OR RPT_DATE > '2021-12-31')
                        ORDER BY 
                        DEAL, 
                        Loan_Type,  
                        FICO_bkt,
                        months_since_deal_clost,
                        RPT_DATE
                        """

    ,   "loans_types_and_FICO_bkt": """
                        SELECT 
                                *
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN default_amt / SUM_BOM_bal ELSE 0 END AS MDR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (default_amt / SUM_BOM_bal), 12)) * 100 ELSE 0 END AS CDR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN prepayment / prepay_den ELSE 0 END AS SMM
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN (1 - POWER(1- (prepayment / prepay_den), 12)) * 100 ELSE 0 END AS CPR
                            ,	CASE WHEN SUM_BOM_bal > 0 THEN SUM_DELIN_amt / SUM_BOM_bal * 100 ELSE 0 END AS delinquency_pct
                        FROM
                        (
                            SELECT
                                    DEAL
                        		,	Loan_Type
                        		,	FICO_bkt
                                ,	months_since_deal_clost
                                ,	RPT_DATE
                                ,	COUNT(*) AS num_obs
                                ,	SUM(default_amount) AS default_amt
                                ,	SUM(BOM_BAL_clean) AS SUM_BOM_bal
                                ,	SUM(CURBALANCE_clean) AS SUM_EOM_bal
                                ,	SUM(BOM_BAL_clean * FICO) / NULLIF(SUM(BOM_BAL_clean),0) AS WA_FICO
                                ,	SUM(BOM_BAL_clean * AI_FICO_AT_LOAN_ORIGINATION) / NULLIF(SUM(BOM_BAL_clean), 0) AS WA_ORIG_FICO
                                ,	SUM(BOM_BAL_clean - CURBALANCE_clean - default_amount - Expected_Prin) AS prepayment
                                ,	SUM(BOM_BAL_clean - Expected_Prin) AS prepay_den
                                ,   SUM(delin_amount) AS SUM_DELIN_amt
                            FROM
                                #cleanTable
                            WHERE DEAL IN ('CMHA19M1', 'CMHA22M1', 'CMHA24M1')
                            GROUP BY 
                            DEAL, 
                        	Loan_Type, 
                        	FICO_bkt,
                            months_since_deal_clost,
                            RPT_DATE
                        ) t
                        WHERE months_since_deal_clost > 0
                        AND (RPT_DATE < '2020-03-01' OR RPT_DATE > '2021-12-31')
                        ORDER BY 
                        DEAL, 
                        Loan_Type, 
                        FICO_bkt,
                        months_since_deal_clost,
                        RPT_DATE
                        """
}

In [3]:
cascade_sql_path = Path.cwd() / "sqls" / "Cascade_all_loans.txt"

if not cascade_sql_path.exists():
    cascade_sql_path = PROJECT_ROOT / "Project Alamo" / "sqls" / "Cascade_all_loans.txt"

temp_table_sql = cascade_sql_path.read_text()

In [4]:
cascade_performance_res_dict = connect_db_temp_table_with_query(temp_table_sql, queries_dict)

g:\YuemengZhang\Deal Opportunities - Clean\databaseHelpers.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  res_df = pd.read_sql(table_creation_query, conn)


In [5]:
cascade_performance = pd.concat([cascade_performance_res_dict['all'], 
                                cascade_performance_res_dict['loans_types_only'], 
                                cascade_performance_res_dict['fico_bkt_only'],
                                cascade_performance_res_dict['loans_types_and_FICO_bkt']
                                ], axis = 0)

In [6]:
display(cascade_performance)

,DEAL,Loan_Type,FICO_bkt,months_since_deal_clost,RPT_DATE,num_obs,default_amt,SUM_BOM_bal,SUM_EOM_bal,WA_FICO,WA_ORIG_FICO,prepayment,prepay_den,SUM_DELIN_amt,MDR,CDR,SMM,CPR,delinquency_pct
0,cmha19m1,ALL,ALL,1,2019-11-25,2054,0.00,1.740118e+08,1.732128e+08,626.822892,625.902700,5.903294e+05,1.738031e+08,2025700.25,0.000000,0.000000,0.003397,4.000564,1.164116
1,cmha19m1,ALL,ALL,2,2019-12-25,2054,0.00,1.732128e+08,1.728011e+08,626.954551,626.032702,2.001463e+05,1.730012e+08,3580767.08,0.000000,0.000000,0.001157,1.379488,2.067265
2,cmha19m1,ALL,ALL,3,2020-01-25,2054,44704.93,1.728011e+08,1.720990e+08,626.900713,625.979040,4.455959e+05,1.725893e+08,6254940.40,0.000259,0.310008,0.002582,3.054576,3.619734
3,cmha19m1,ALL,ALL,4,2020-02-25,2054,144607.01,1.720990e+08,1.714252e+08,626.867908,625.945322,3.159602e+05,1.718858e+08,3640352.38,0.000840,1.003659,0.001838,2.183673,2.115267
4,cmha19m1,ALL,ALL,27,2022-01-25,2054,436061.88,1.435385e+08,1.418112e+08,625.936257,625.344951,1.054929e+06,1.433022e+08,5322937.94,0.003038,3.585233,0.007362,8.484840,3.708369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1758,cmha24m1,Land Home,Other,15,2025-12-25,1,0.00,2.683811e+05,2.681524e+05,775.000000,775.000000,-1.132489e+00,2.681513e+05,0.00,0.000000,0.000000,-0.000004,-0.005068,0.000000
1759,cmha24m1,Land Home,Other,16,2026-01-25,1,0.00,2.681524e+05,2.679224e+05,775.000000,775.000000,-1.141491e+00,2.679212e+05,0.00,0.000000,0.000000,-0.000004,-0.005113,0.000000
1760,cmha24m1,Land Home,Other,17,2026-02-25,1,0.00,2.679224e+05,2.676909e+05,775.000000,775.000000,-1.149019e+00,2.676898e+05,0.00,0.000000,0.000000,-0.000004,-0.005151,0.000000
1761,cmha24m1,Land Home,Other,18,2026-03-25,1,0.00,2.676909e+05,2.674580e+05,775.000000,775.000000,-1.155122e+00,2.674569e+05,0.00,0.000000,0.000000,-0.000004,-0.005183,0.000000


In [7]:
perf = cascade_performance.copy()

mob_col = "months_since_deal_clost"
deal_col = "DEAL"
loan_type_col = "Loan_Type"
fico_col = "FICO_bkt"
# loan_types = sorted(perf[loan_type_col].dropna().unique())
# fico_bkts = sorted(perf[fico_col].dropna().unique())
loan_types = ['Chattel', 'Land Home', 'ALL']
fico_bkts = ['1_<620 OR No FICO',
            '2_620 - 649',
            '3_650 - 679',
            '4_680 - 700',
            '5_700 - 740',
            '6_740 - 775',
            '7_>775',
            'ALL']

metrics = ["CDR", "CPR", "delinquency_pct", "num_obs"]

In [8]:
pivot_tables = {}

for loan_type in loan_types:
    for fico_bkt in fico_bkts:
        filtered = perf[
            (perf[loan_type_col] == loan_type) &
            (perf[fico_col] == fico_bkt)
        ]

        if filtered.empty:
            continue

        for metric in metrics:
            table = (
                filtered
                .pivot_table(
                    index=mob_col,
                    columns=deal_col,
                    values=metric,
                    aggfunc="first"
                )
                .sort_index()
            )

            pivot_tables[(loan_type, fico_bkt, metric)] = table

In [9]:
# loan_type = "Chattel"
# metric = "CDR"

# for fico_bkt in fico_bkts:
#     table = pivot_tables[(loan_type, fico_bkt, metric)]

#     plot_df = table.reset_index()

#     fig = go.Figure()

#     for col in table.columns:
#         fig.add_trace(
#             go.Scatter(
#                 x=plot_df["months_since_deal_clost"],
#                 y=plot_df[col],
#                 mode="lines+markers",
#                 name=col,
#                 hovertemplate=(
#                     "Deal Age: %{x}<br>"
#                     f"{col}: " + "%{y:.1f}<extra></extra>"
#                 ),
#             )
#         )

#     fig.update_layout(
#         title=f"{metric} | {loan_type} | {fico_bkt}",
#         hovermode="x unified",
#         width=1000,
#         height=550,
#     )

#     fig.update_xaxes(title_text="Deal Age")
#     fig.update_yaxes(title_text=metric, tickformat=".1f")

#     fig.show()

In [10]:
# sorted(cascade_performance["Loan_Type"].dropna().unique())

In [11]:
import ast
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

for _old_widget in globals().get("_cascade_ui_widgets", []):
    try:
        _old_widget.close()
    except Exception:
        pass

for _old_name in [
    "loan_type_dd", "fico_dd", "metric_dd", "rolling_dd", "y_max_txt",
    "base_case_txt", "save_base_btn", "clear_base_btn",
    "show_base_btn", "export_base_btn", "status_out", "fig",
]:
    _old_widget = globals().get(_old_name)
    if hasattr(_old_widget, "close"):
        try:
            _old_widget.close()
        except Exception:
            pass

base_case_store = globals().get("base_case_store", {})
base_case_export_path = (
    cascade_sql_path.parent.parent / "Cascade_base_cases.xlsx"
    if "cascade_sql_path" in globals()
    else Path("Cascade_base_cases.xlsx")
)
mob_col_name = "months_since_deal_clost"
_cascade_ui_run_id = object()
globals()["_cascade_ui_active_run_id"] = _cascade_ui_run_id

def _is_current_ui_run(run_id=_cascade_ui_run_id):
    return globals().get("_cascade_ui_active_run_id") is run_id

loan_type_dd = widgets.Dropdown(
    options=loan_types,
    description="Loan Type:"
)

fico_dd = widgets.Dropdown(
    options=fico_bkts,
    description="FICO:"
)

metric_dd = widgets.Dropdown(
    options=["CDR", "CPR", "delinquency_pct"],
    description="Metric:"
)

rolling_dd = widgets.Dropdown(
    options=[("1M", 1), ("3M Rolling Avg", 3), ("6M Rolling Avg", 6)],
    description="Rolling:"
)

y_max_txt = widgets.Text(
    value="",
    placeholder="Auto",
    description="Y Max:",
    layout=widgets.Layout(width="160px"),
)

base_case_txt = widgets.Textarea(
    value="",
    placeholder="Paste values, MOB/value pairs, or [3] * 3 + [5] * 100",
    description="Base Case:",
    layout=widgets.Layout(width="1000px", height="80px"),
)

save_base_btn = widgets.Button(description="Save Base Case", button_style="success")
clear_base_btn = widgets.Button(description="Clear Base Case")
show_base_btn = widgets.Button(description="Show Saved Base Cases")
export_base_btn = widgets.Button(description="Export Base Cases")

status_out = widgets.Output()
fig = go.FigureWidget()

def current_base_key():
    return (loan_type_dd.value, fico_dd.value, metric_dd.value)

def build_plot_table(key, rolling_window):
    if key not in pivot_tables:
        return None

    table = pivot_tables[key].copy()
    if rolling_window > 1:
        table = table.rolling(window=rolling_window, min_periods=1).mean()
        table = table.apply(trim_last_n, n=rolling_window - 1)

    return table

def _to_float(value):
    try:
        return float(str(value).strip().replace("%", ""))
    except (TypeError, ValueError):
        return None

def _rows_from_values(values, x_values):
    rows = []
    for i, value in enumerate(values):
        value = _to_float(value)
        if value is None:
            continue

        mob = x_values.iloc[i] if i < len(x_values) else i + 1
        rows.append({mob_col_name: mob, "Base_Case": value})

    return rows

def _eval_base_case_expr(node):
    if isinstance(node, ast.Expression):
        return _eval_base_case_expr(node.body)

    if isinstance(node, ast.List):
        return [_eval_base_case_expr(item) for item in node.elts]

    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        value = _eval_base_case_expr(node.operand)
        if isinstance(value, (int, float)):
            return -value

    if isinstance(node, ast.BinOp):
        left = _eval_base_case_expr(node.left)
        right = _eval_base_case_expr(node.right)

        if isinstance(node.op, ast.Add) and isinstance(left, list) and isinstance(right, list):
            return left + right

        if isinstance(node.op, ast.Mult):
            if isinstance(left, list) and isinstance(right, int):
                return left * right
            if isinstance(right, list) and isinstance(left, int):
                return right * left

    raise ValueError("Unsupported base case expression")

def _parse_base_case_expr(text):
    try:
        values = _eval_base_case_expr(ast.parse(text, mode="eval"))
    except (SyntaxError, ValueError):
        return None

    return values if isinstance(values, list) else None

def parse_base_case_text(text, x_values):
    text = text.strip()
    expr_values = _parse_base_case_expr(text)
    if expr_values is not None:
        return _rows_from_values(expr_values, x_values)

    lines = [line.strip() for line in text.replace("\r", "\n").split("\n") if line.strip()]
    rows = []

    if not lines:
        return rows

    if len(lines) == 1:
        values = [_to_float(token) for token in re.split(r"[\t, ]+", lines[0]) if token.strip()]
        values = [value for value in values if value is not None]
        return _rows_from_values(values, x_values)

    for line in lines:
        values = [_to_float(token) for token in re.split(r"[\t, ]+", line) if token.strip()]
        values = [value for value in values if value is not None]
        if len(values) >= 2:
            rows.append({mob_col_name: values[0], "Base_Case": values[-1]})
        elif len(values) == 1:
            i = len(rows)
            mob = x_values.iloc[i] if i < len(x_values) else i + 1
            rows.append({mob_col_name: mob, "Base_Case": values[0]})

    return rows

def format_base_case_text(rows):
    return "\n".join(f"{row[mob_col_name]}\t{row['Base_Case']}" for row in rows)

def load_base_case_text():
    base_case_txt.value = format_base_case_text(base_case_store.get(current_base_key(), []))

def base_case_store_to_df():
    rows = []
    for (loan_type, fico_bkt, metric), base_rows in base_case_store.items():
        for row in base_rows:
            rows.append({
                "Loan_Type": loan_type,
                "FICO_bkt": fico_bkt,
                "Metric": metric,
                mob_col_name: row[mob_col_name],
                "Base_Case": row["Base_Case"],
            })

    columns = ["Loan_Type", "FICO_bkt", "Metric", mob_col_name, "Base_Case"]
    if not rows:
        return pd.DataFrame(columns=columns)

    return pd.DataFrame(rows, columns=columns).sort_values(columns[:-1]).reset_index(drop=True)

def refresh_saved_base_cases_df():
    saved_base_cases_df = base_case_store_to_df()
    globals()["saved_base_cases_df"] = saved_base_cases_df
    return saved_base_cases_df

def update_chart(*args):
    if not _is_current_ui_run():
        return

    status_out.clear_output(wait=True)

    loan_type = loan_type_dd.value
    fico_bkt = fico_dd.value
    metric = metric_dd.value
    rolling_window = rolling_dd.value
    y_max = _to_float(y_max_txt.value)
    key = current_base_key()

    with fig.batch_update():
        fig.data = []

        table = build_plot_table(key, rolling_window)
        if table is None:
            fig.update_layout(
                title=f"No data for {metric} | {loan_type} | {fico_bkt}",
                hovermode="x unified",
                width=1000,
                height=550,
            )
            with status_out:
                print(f"No data for {metric} | {loan_type} | {fico_bkt}")
            return

        plot_df = table.reset_index()

        for col in table.columns:
            series_label = col if rolling_window == 1 else f"{col} ({rolling_window}M avg)"
            fig.add_scatter(
                x=plot_df[mob_col_name],
                y=plot_df[col],
                mode="lines+markers",
                name=series_label,
            )

        base_rows = base_case_store.get(key, [])
        if base_rows:
            base_df = pd.DataFrame(base_rows)
            fig.add_scatter(
                x=base_df[mob_col_name],
                y=base_df["Base_Case"],
                mode="lines+markers",
                name="Base Case",
                line=dict(color="black", width=3, dash="dash"),
                marker=dict(size=7),
            )

        title_suffix = "" if rolling_window == 1 else f" | {rolling_window}M Rolling Avg"
        fig.update_layout(
            title=f"{metric} | {loan_type} | {fico_bkt}{title_suffix}",
            hovermode="x unified",
            width=1000,
            height=550,
        )
        fig.update_xaxes(title_text="Deal Age")
        if y_max is not None and y_max > 0:
            fig.update_yaxes(title_text=metric, tickformat=".1f", range=[0, y_max], autorange=False)
        else:
            fig.update_yaxes(title_text=metric, tickformat=".1f", autorange=True)

def on_segment_change(*args):
    if not _is_current_ui_run():
        return

    load_base_case_text()
    update_chart()

def save_base_case(_):
    if not _is_current_ui_run():
        return

    key = current_base_key()
    table = build_plot_table(key, rolling_dd.value)
    if table is None:
        update_chart()
        return

    rows = parse_base_case_text(base_case_txt.value, table.reset_index()[mob_col_name])
    if rows:
        base_case_store[key] = rows
    else:
        base_case_store.pop(key, None)

    saved_df = refresh_saved_base_cases_df()
    update_chart()
    with status_out:
        print(f"Saved {len(rows)} base case points for {key[0]} | {key[1]} | {key[2]}")
        print(f"saved_base_cases_df now has {len(saved_df)} rows.")
        # display(saved_df)

def clear_base_case(_):
    if not _is_current_ui_run():
        return

    key = current_base_key()
    base_case_store.pop(key, None)
    base_case_txt.value = ""
    saved_df = refresh_saved_base_cases_df()
    update_chart()
    with status_out:
        print(f"Cleared base case for {key[0]} | {key[1]} | {key[2]}")
        print(f"saved_base_cases_df now has {len(saved_df)} rows.")
        display(saved_df)

def show_saved_base_cases(_):
    if not _is_current_ui_run():
        return

    saved_df = refresh_saved_base_cases_df()
    status_out.clear_output(wait=True)
    with status_out:
        if saved_df.empty:
            print("No saved base cases yet.")
        else:
            print(f"saved_base_cases_df has {len(saved_df)} rows.")
            display(saved_df)

def export_base_cases(_):
    if not _is_current_ui_run():
        return

    base_case_df = refresh_saved_base_cases_df()
    status_out.clear_output(wait=True)
    with status_out:
        if base_case_df.empty:
            print("No saved base cases to export.")
            return

        try:
            base_case_df.to_excel(base_case_export_path, index=False)
            print(f"Exported base cases to {base_case_export_path}")
        except Exception as exc:
            fallback_path = base_case_export_path.with_suffix(".csv")
            base_case_df.to_csv(fallback_path, index=False)
            print(f"Excel export failed: {exc}")
            print(f"Exported base cases to {fallback_path} instead.")

for w in [loan_type_dd, fico_dd, metric_dd]:
    w.observe(on_segment_change, names="value")
rolling_dd.observe(update_chart, names="value")
y_max_txt.observe(update_chart, names="value")
save_base_btn.on_click(save_base_case)
clear_base_btn.on_click(clear_base_case)
# show_base_btn.on_click(show_saved_base_cases)
export_base_btn.on_click(export_base_cases)

_cascade_ui_widgets = [
    loan_type_dd, fico_dd, metric_dd, rolling_dd, y_max_txt,
    base_case_txt, save_base_btn, clear_base_btn,
    # show_base_btn, 
    export_base_btn, status_out, fig,
]
globals()["_cascade_ui_widgets"] = _cascade_ui_widgets

refresh_saved_base_cases_df()
load_base_case_text()
update_chart()
display(widgets.VBox([
    widgets.HBox([loan_type_dd, fico_dd, metric_dd, rolling_dd, y_max_txt]),
    base_case_txt,
    widgets.HBox([save_base_btn, clear_base_btn, 
                #   show_base_btn, 
                  export_base_btn]),
    status_out,
    fig,
]))

In [12]:
display(saved_base_cases_df)

,Loan_Type,FICO_bkt,Metric,months_since_deal_clost,Base_Case


In [13]:
# saved_base_cases_df.to_excel(base_case_export_path, index=False)